# Part 1B: Bresenham's Line Algorithm

## The Elegant Solution

Welcome back! In Part 1A, we discovered how to draw lines by stepping through the appropriate dimension (x for gentle lines, y for steep lines). Our solution works, but it has a significant limitation: it uses floating-point arithmetic, which was computationally expensive in the 1960s and can still be slower than integer operations on modern hardware.

In this notebook, we're going to discover one of the most elegant algorithms in computer graphics history: **Bresenham's line algorithm**, invented by Jack E. Bresenham in 1962 while working at IBM.

What makes this algorithm special? It draws perfect lines using:
- **Only integer arithmetic** (no floating-point operations)
- **Only addition and subtraction** (no multiplication or division)
- **A simple decision at each step** (should we move diagonally or straight?)

The algorithm is so efficient that variations of it are still used in graphics hardware today, more than 60 years after its invention. But beyond its efficiency, Bresenham's algorithm is also intellectually beautiful—it solves a geometric problem using only integer arithmetic through a clever insight about error accumulation.

We'll build this algorithm from the ground up, understanding each step. By the end, you'll appreciate why this is considered a classic of computer science.

## Section 1: Setting Up

Let's start by importing our libraries and bringing in our helper functions from Part 1A.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
import time
from IPython.display import clear_output

# Set up matplotlib
plt.rcParams['figure.figsize'] = (10, 10)
plt.rcParams['image.cmap'] = 'gray'
plt.rcParams['image.interpolation'] = 'nearest'

def create_canvas(width=50, height=50):
    """Create a blank canvas."""
    return np.zeros((height, width))

def show_canvas(canvas, title="Canvas"):
    """Display a canvas with proper pixel visualization."""
    plt.figure(figsize=(8, 8))
    plt.imshow(canvas, cmap='gray', vmin=0, vmax=1)
    plt.title(title, fontsize=14, pad=15)
    plt.grid(True, alpha=0.3, linewidth=0.5)
    plt.show()

print("Ready to explore Bresenham's algorithm!")

## Section 2: The Core Insight

### The Problem We're Solving

Imagine we're drawing a line from left to right with a gentle positive slope (0 < slope < 1). As we move from left to right, at each x-coordinate, we need to decide:

**Should we plot the pixel at the same y-level, or should we move up one pixel?**

In other words, as we step rightward (x increases), should y stay the same or increment by 1?

### Visualizing the Decision

Let's draw a diagram to understand this better. At each step, we're at some pixel (x, y), and the true mathematical line passes somewhere through or near that pixel. We need to decide: is the next pixel (x+1, y) or (x+1, y+1)?

In [ ]:
# Visualize the decision problem
fig, ax = plt.subplots(figsize=(10, 6))

# Draw a grid
for i in range(8):
    ax.axhline(y=i, color='lightgray', linewidth=0.5)
    ax.axvline(x=i, color='lightgray', linewidth=0.5)

# Draw the mathematical line (y = 0.4x)
x_line = np.linspace(0, 7, 100)
y_line = 0.4 * x_line
ax.plot(x_line, y_line, 'b-', linewidth=2, label='Mathematical line (y=0.4x)')

# Show current pixel
current_x, current_y = 3, 1
ax.add_patch(plt.Rectangle((current_x, current_y), 1, 1, 
                           fill=True, color='green', alpha=0.5,
                           label='Current pixel'))

# Show two possible next pixels
ax.add_patch(plt.Rectangle((current_x+1, current_y), 1, 1,
                           fill=True, color='orange', alpha=0.5,
                           label='Option 1: Stay at y'))
ax.add_patch(plt.Rectangle((current_x+1, current_y+1), 1, 1,
                           fill=True, color='red', alpha=0.5,
                           label='Option 2: Increment y'))

# Show where the true line passes
true_y_at_next_x = 0.4 * (current_x + 1.5)  # Center of next column
ax.plot(current_x + 1.5, true_y_at_next_x, 'b*', markersize=15)
ax.text(current_x + 1.5, true_y_at_next_x + 0.3, 'True line\nposition', 
        ha='center', fontsize=10, color='blue')

ax.set_xlim(0, 7)
ax.set_ylim(0, 5)
ax.set_aspect('equal')
ax.legend(loc='upper left', fontsize=10)
ax.set_title('The Decision Problem: Which Pixel Should We Choose?', fontsize=13, pad=15)
ax.set_xlabel('x coordinate', fontsize=11)
ax.set_ylabel('y coordinate', fontsize=11)

plt.tight_layout()
plt.show()

print("At each step, we need to decide:")
print("- Orange pixel: keep y the same")
print("- Red pixel: increment y by 1")
print("\nWhich one is closer to where the true line passes?")

### The Key Question

How can we make this decision using only integer arithmetic?

Bresenham's brilliant insight was to track the **error** between where we've drawn pixels and where the true mathematical line would be. When this error gets large enough, we increment y. Otherwise, we keep y the same.

Let's develop this idea step by step.

## Section 3: Building the Algorithm

### Step 1: Understanding Error

Let's say we're drawing a line with slope m (where 0 < m < 1). Starting at point (x₀, y₀), the true y-coordinate when we've moved to x is:

```
y_true = y₀ + m × (x - x₀)
```

But we're actually drawing at integer y-coordinates. Let's call our current drawing position y_drawn. The error is:

```
error = y_true - y_drawn
```

When error gets close to 0.5, it means the true line has climbed about halfway to the next pixel row. That's when we should increment y!

### Step 2: Avoiding Fractions

The problem is that m is typically a fraction, and comparing error to 0.5 involves fractions. Can we reformulate this to use only integers?

Here's the trick: instead of tracking error directly, we track **2 × error × dx**, where dx = x₁ - x₀.

Why? Let's see:

```
m = dy / dx    (where dy = y₁ - y₀, dx = x₁ - x₀)

When we move one pixel right (from x to x+1):
error increases by m = dy / dx

Instead, track: decision_variable = 2 × error × dx

When we move right:
decision_variable increases by 2 × dy

When we should increment y (error ≥ 0.5):
decision_variable ≥ dx
```

Now we're comparing integers! Let's write this out in pseudocode.

### Pseudocode: Bresenham's Algorithm (First Octant)

Let's write pseudocode for lines in the "first octant" (0 < slope < 1, moving right and up):

```
FUNCTION draw_line_bresenham_simple(canvas, x0, y0, x1, y1):
    # Calculate differences
    dx = x1 - x0  # How far right we're going
    dy = y1 - y0  # How far up we're going
    
    # Initialize decision variable
    # This tracks 2 × error × dx
    decision = 2 × dy - dx
    
    # Initialize drawing position
    y = y0
    
    # Step through x coordinates
    FOR x FROM x0 TO x1:
        # Draw current pixel
        SET canvas[y, x] = 1
        
        # Should we move up?
        IF decision > 0:
            y = y + 1                    # Move up
            decision = decision - 2×dx    # Adjust decision variable
        
        # Always move right (handled by FOR loop)
        decision = decision + 2×dy        # Accumulate error
    
    RETURN canvas
```

Let's implement this and see it work!

In [ ]:
def draw_line_bresenham_simple(canvas, x0, y0, x1, y1):
    """
    Bresenham's line algorithm for first octant (0 < slope < 1).
    
    This version only works for lines that:
    - Go from left to right (x1 > x0)
    - Go upward or stay level (y1 >= y0)
    - Have gentle slope (dy <= dx)
    
    Parameters:
    -----------
    canvas : numpy.ndarray
        Canvas to draw on
    x0, y0 : int
        Starting point
    x1, y1 : int
        Ending point
    """
    # Calculate differences
    dx = x1 - x0
    dy = y1 - y0
    
    # Initialize decision variable
    decision = 2 * dy - dx
    
    # Start at y0
    y = y0
    
    print(f"Drawing line from ({x0}, {y0}) to ({x1}, {y1})")
    print(f"dx = {dx}, dy = {dy}")
    print(f"Initial decision variable: {decision}")
    print("\nStep-by-step:")
    
    # Step through x coordinates
    for x in range(x0, x1 + 1):
        # Draw pixel
        if 0 <= y < canvas.shape[0] and 0 <= x < canvas.shape[1]:
            canvas[y, x] = 1
        
        # Print what we're doing
        print(f"  x={x}, y={y}, decision={decision}", end="")
        
        # Decide whether to increment y
        if decision > 0:
            y += 1
            decision -= 2 * dx
            print(f" → Move up! New y={y}")
        else:
            print(f" → Stay at same y")
        
        # Always increment decision (accumulate error)
        decision += 2 * dy

# Test with a simple line
canvas = create_canvas(20, 20)
draw_line_bresenham_simple(canvas, 2, 2, 15, 8)
show_canvas(canvas, "Bresenham's Algorithm (First Octant)")

print("\n✓ Notice: Only integer operations! No floating-point math!")

### Understanding the Decision Variable

Let's visualize what the decision variable is actually tracking. We'll plot it alongside the line being drawn.

In [ ]:
def visualize_bresenham_decision(x0, y0, x1, y1):
    """
    Visualize how the decision variable evolves as we draw the line.
    """
    dx = x1 - x0
    dy = y1 - y0
    decision = 2 * dy - dx
    y = y0
    
    # Track decision variable and y values
    x_values = []
    y_values = []
    decision_values = []
    
    for x in range(x0, x1 + 1):
        x_values.append(x)
        y_values.append(y)
        decision_values.append(decision)
        
        if decision > 0:
            y += 1
            decision -= 2 * dx
        decision += 2 * dy
    
    # Create visualization
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
    
    # Top plot: The line being drawn
    canvas = create_canvas(20, 12)
    for i in range(len(x_values)):
        if 0 <= y_values[i] < 12 and 0 <= x_values[i] < 20:
            canvas[y_values[i], x_values[i]] = 1
    
    ax1.imshow(canvas, cmap='gray', vmin=0, vmax=1, aspect='auto')
    ax1.set_title('The Line Being Drawn', fontsize=12)
    ax1.grid(True, alpha=0.3)
    ax1.set_ylabel('y coordinate')
    
    # Bottom plot: Decision variable over time
    ax2.plot(x_values, decision_values, 'b-o', linewidth=2, markersize=6)
    ax2.axhline(y=0, color='red', linestyle='--', linewidth=2, 
                label='decision = 0 (threshold)')
    ax2.fill_between(x_values, 0, decision_values, 
                     where=[d > 0 for d in decision_values],
                     alpha=0.3, color='orange', label='decision > 0 (increment y)')
    ax2.set_title('Decision Variable Evolution', fontsize=12)
    ax2.set_xlabel('x coordinate', fontsize=11)
    ax2.set_ylabel('Decision variable', fontsize=11)
    ax2.legend(loc='best')
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\nObservations:")
    print("- When decision variable crosses above 0, we increment y")
    print("- The decision variable oscillates around 0")
    print("- This keeps our drawn line close to the true mathematical line")

visualize_bresenham_decision(2, 2, 18, 10)

## Section 4: Generalizing to All Octants

Our simple version only works for lines in the "first octant" (gentle positive slopes going right and up). But lines can go in any direction! We need to handle:

- Lines going left (x₁ < x₀)
- Lines going down (y₁ < y₀)
- Steep lines (|dy| > |dx|)

### The Eight Octants

We can divide 2D space into eight regions based on:
1. Whether dx or dy is larger (gentle vs. steep)
2. The sign of dx (left vs. right)
3. The sign of dy (up vs. down)

Let's visualize these octants:

In [ ]:
# Visualize the eight octants
fig, ax = plt.subplots(figsize=(10, 10))

center = (25, 25)
length = 15

# Define eight directions with labels
octants = [
    (length, 0, "East\n(gentle right)", 0),
    (length, length//2, "ENE\n(gentle right-up)", 1),
    (length//2, length, "NNE\n(steep up-right)", 2),
    (0, length, "North\n(steep up)", 3),
    (-length//2, length, "NNW\n(steep up-left)", 4),
    (-length, length//2, "WNW\n(gentle left-up)", 5),
    (-length, 0, "West\n(gentle left)", 6),
    (-length, -length//2, "WSW\n(gentle left-down)", 7)
]

colors = plt.cm.tab10(np.linspace(0, 1, 8))

for (dx, dy, label, octant) in octants:
    end = (center[0] + dx, center[1] + dy)
    ax.annotate('', xy=end, xytext=center,
               arrowprops=dict(arrowstyle='->', lw=3, color=colors[octant]))
    
    # Position label
    label_offset = 1.3
    label_x = center[0] + dx * label_offset
    label_y = center[1] + dy * label_offset
    ax.text(label_x, label_y, f"Octant {octant}\n{label}",
           ha='center', va='center', fontsize=9,
           bbox=dict(boxstyle='round', facecolor=colors[octant], alpha=0.3))

ax.plot(center[0], center[1], 'ko', markersize=10)
ax.set_xlim(0, 50)
ax.set_ylim(0, 50)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.set_title('The Eight Octants', fontsize=14, pad=15)
ax.set_xlabel('x coordinate', fontsize=11)
ax.set_ylabel('y coordinate', fontsize=11)

plt.tight_layout()
plt.show()

print("To handle all octants, we need to:")
print("1. Determine which octant the line is in")
print("2. Adjust our algorithm's stepping and decision-making accordingly")
print("3. Use the same core logic, just with different parameters")

### Complete Pseudocode: All Octants

Here's the full algorithm that handles lines in any direction:

```
FUNCTION draw_line_bresenham(canvas, x0, y0, x1, y1):
    
    # Calculate differences
    dx = abs(x1 - x0)
    dy = abs(y1 - y0)
    
    # Determine step directions
    IF x0 < x1:
        x_step = 1   # Moving right
    ELSE:
        x_step = -1  # Moving left
    
    IF y0 < y1:
        y_step = 1   # Moving up
    ELSE:
        y_step = -1  # Moving down
    
    # Determine if line is steep
    IF dx > dy:
        # Gentle line: step through x, occasionally increment y
        decision = 2 × dy - dx
        
        WHILE x0 ≠ x1:
            SET canvas[y0, x0] = 1
            
            IF decision > 0:
                y0 = y0 + y_step
                decision = decision - 2×dx
            
            x0 = x0 + x_step
            decision = decision + 2×dy
    
    ELSE:
        # Steep line: step through y, occasionally increment x
        decision = 2 × dx - dy
        
        WHILE y0 ≠ y1:
            SET canvas[y0, x0] = 1
            
            IF decision > 0:
                x0 = x0 + x_step
                decision = decision - 2×dy
            
            y0 = y0 + y_step
            decision = decision + 2×dx
    
    # Draw final pixel
    SET canvas[y0, x0] = 1
    
    RETURN canvas
```

In [ ]:
def draw_line_bresenham(canvas, x0, y0, x1, y1):
    """
    Complete Bresenham's line algorithm for all octants.
    Uses only integer arithmetic.
    
    Parameters:
    -----------
    canvas : numpy.ndarray
        Canvas to draw on
    x0, y0 : int
        Starting point
    x1, y1 : int
        Ending point
    """
    # Calculate absolute differences
    dx = abs(x1 - x0)
    dy = abs(y1 - y0)
    
    # Determine step directions
    x_step = 1 if x0 < x1 else -1
    y_step = 1 if y0 < y1 else -1
    
    # Determine if line is steep
    if dx > dy:
        # Gentle line: step through x
        decision = 2 * dy - dx
        
        while x0 != x1:
            if 0 <= y0 < canvas.shape[0] and 0 <= x0 < canvas.shape[1]:
                canvas[y0, x0] = 1
            
            if decision > 0:
                y0 += y_step
                decision -= 2 * dx
            
            x0 += x_step
            decision += 2 * dy
    else:
        # Steep line: step through y
        decision = 2 * dx - dy
        
        while y0 != y1:
            if 0 <= y0 < canvas.shape[0] and 0 <= x0 < canvas.shape[1]:
                canvas[y0, x0] = 1
            
            if decision > 0:
                x0 += x_step
                decision -= 2 * dy
            
            y0 += y_step
            decision += 2 * dx
    
    # Draw final pixel
    if 0 <= y0 < canvas.shape[0] and 0 <= x0 < canvas.shape[1]:
        canvas[y0, x0] = 1

# Test with lines in all eight octants
canvas = create_canvas(50, 50)
center_x, center_y = 25, 25
length = 15

directions = [
    (length, 0),
    (length, length//2),
    (length//2, length),
    (0, length),
    (-length//2, length),
    (-length, length//2),
    (-length, 0),
    (-length, -length//2)
]

for dx, dy in directions:
    draw_line_bresenham(canvas, center_x, center_y, 
                       center_x + dx, center_y + dy)

show_canvas(canvas, "Bresenham's Algorithm - All Octants")

print("Success! The algorithm works in all directions.")
print("All using only integer arithmetic!")

## Section 5: Extensions and Variations

Now that we have the core algorithm working, let's explore some useful extensions.

### Extension 1: Thick Lines

Sometimes we want lines that are more than one pixel thick. How can we do this?

A simple approach: for each pixel we'd normally draw, also draw the pixels immediately above and below it (for gentle lines) or left and right (for steep lines).

In [ ]:
def draw_thick_line(canvas, x0, y0, x1, y1, thickness=3):
    """
    Draw a thick line using Bresenham's algorithm.
    
    Parameters:
    -----------
    canvas : numpy.ndarray
        Canvas to draw on
    x0, y0 : int
        Starting point
    x1, y1 : int
        Ending point
    thickness : int
        Line thickness in pixels
    """
    dx = abs(x1 - x0)
    dy = abs(y1 - y0)
    x_step = 1 if x0 < x1 else -1
    y_step = 1 if y0 < y1 else -1
    
    offset = thickness // 2
    
    if dx > dy:
        # Gentle line: thicken vertically
        decision = 2 * dy - dx
        
        while x0 != x1:
            # Draw thick pixel
            for dy_offset in range(-offset, offset + 1):
                y_draw = y0 + dy_offset
                if 0 <= y_draw < canvas.shape[0] and 0 <= x0 < canvas.shape[1]:
                    canvas[y_draw, x0] = 1
            
            if decision > 0:
                y0 += y_step
                decision -= 2 * dx
            
            x0 += x_step
            decision += 2 * dy
    else:
        # Steep line: thicken horizontally
        decision = 2 * dx - dy
        
        while y0 != y1:
            # Draw thick pixel
            for dx_offset in range(-offset, offset + 1):
                x_draw = x0 + dx_offset
                if 0 <= y0 < canvas.shape[0] and 0 <= x_draw < canvas.shape[1]:
                    canvas[y0, x_draw] = 1
            
            if decision > 0:
                x0 += x_step
                decision -= 2 * dy
            
            y0 += y_step
            decision += 2 * dx
    
    # Draw final thick pixel
    for dy_offset in range(-offset, offset + 1):
        for dx_offset in range(-offset, offset + 1):
            y_draw = y0 + dy_offset
            x_draw = x0 + dx_offset
            if 0 <= y_draw < canvas.shape[0] and 0 <= x_draw < canvas.shape[1]:
                canvas[y_draw, x_draw] = 1

# Compare thin and thick lines
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

for idx, thickness in enumerate([1, 3, 5]):
    canvas = create_canvas(40, 40)
    
    if thickness == 1:
        draw_line_bresenham(canvas, 5, 5, 35, 30)
    else:
        draw_thick_line(canvas, 5, 5, 35, 30, thickness)
    
    axes[idx].imshow(canvas, cmap='gray', vmin=0, vmax=1)
    axes[idx].set_title(f'Thickness = {thickness} pixel{"s" if thickness > 1 else ""}', 
                       fontsize=12)
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Thick lines are useful for:")
print("- User interface elements")
print("- Drawing tools")
print("- Emphasis in diagrams")

### Extension 2: Dotted and Dashed Lines

How about creating patterns? We can modify our algorithm to skip certain pixels, creating dotted or dashed lines.

In [ ]:
def draw_patterned_line(canvas, x0, y0, x1, y1, pattern=[1, 1, 1, 0, 0]):
    """
    Draw a line with a repeating pattern.
    
    Parameters:
    -----------
    canvas : numpy.ndarray
        Canvas to draw on
    x0, y0 : int
        Starting point
    x1, y1 : int
        Ending point
    pattern : list
        Pattern where 1 = draw pixel, 0 = skip pixel
        Example: [1,1,1,0,0] = dash (3 on, 2 off)
    """
    dx = abs(x1 - x0)
    dy = abs(y1 - y0)
    x_step = 1 if x0 < x1 else -1
    y_step = 1 if y0 < y1 else -1
    
    pattern_index = 0
    
    if dx > dy:
        decision = 2 * dy - dx
        
        while x0 != x1:
            # Draw according to pattern
            if pattern[pattern_index % len(pattern)]:
                if 0 <= y0 < canvas.shape[0] and 0 <= x0 < canvas.shape[1]:
                    canvas[y0, x0] = 1
            
            pattern_index += 1
            
            if decision > 0:
                y0 += y_step
                decision -= 2 * dx
            
            x0 += x_step
            decision += 2 * dy
    else:
        decision = 2 * dx - dy
        
        while y0 != y1:
            # Draw according to pattern
            if pattern[pattern_index % len(pattern)]:
                if 0 <= y0 < canvas.shape[0] and 0 <= x0 < canvas.shape[1]:
                    canvas[y0, x0] = 1
            
            pattern_index += 1
            
            if decision > 0:
                x0 += x_step
                decision -= 2 * dy
            
            y0 += y_step
            decision += 2 * dx
    
    # Draw final pixel if pattern allows
    if pattern[pattern_index % len(pattern)]:
        if 0 <= y0 < canvas.shape[0] and 0 <= x0 < canvas.shape[1]:
            canvas[y0, x0] = 1

# Test different patterns
fig, axes = plt.subplots(2, 2, figsize=(12, 12))
patterns = [
    ([1, 1, 1, 1, 1], "Solid"),
    ([1, 0], "Dotted"),
    ([1, 1, 1, 0, 0], "Dashed"),
    ([1, 1, 1, 0, 1, 0], "Dash-dot")
]

for idx, (pattern, name) in enumerate(patterns):
    row = idx // 2
    col = idx % 2
    
    canvas = create_canvas(40, 40)
    draw_patterned_line(canvas, 5, 10, 35, 30, pattern)
    
    axes[row, col].imshow(canvas, cmap='gray', vmin=0, vmax=1)
    axes[row, col].set_title(f'{name} Line\nPattern: {pattern}', fontsize=11)
    axes[row, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Patterned lines are used in:")
print("- Technical drawings (to show hidden edges)")
print("- Data visualization (to distinguish multiple lines)")
print("- User interface design (to show boundaries)")

### Extension 3: Drawing Shapes

Let's use our line-drawing algorithm to create more complex shapes. We'll build some classic geometric forms.

In [ ]:
def draw_polygon(canvas, points, closed=True):
    """
    Draw a polygon by connecting points.
    
    Parameters:
    -----------
    canvas : numpy.ndarray
        Canvas to draw on
    points : list of tuples
        Vertices as (x, y) pairs
    closed : bool
        Whether to connect last point back to first
    """
    for i in range(len(points) - 1):
        x0, y0 = points[i]
        x1, y1 = points[i + 1]
        draw_line_bresenham(canvas, x0, y0, x1, y1)
    
    if closed:
        x0, y0 = points[-1]
        x1, y1 = points[0]
        draw_line_bresenham(canvas, x0, y0, x1, y1)

import math

def create_regular_polygon(center_x, center_y, radius, n_sides):
    """
    Create vertices for a regular polygon.
    
    Parameters:
    -----------
    center_x, center_y : int
        Center coordinates
    radius : int
        Radius from center to vertices
    n_sides : int
        Number of sides
    
    Returns:
    --------
    list of tuples
        Vertices as (x, y) pairs
    """
    points = []
    for i in range(n_sides):
        angle = 2 * math.pi * i / n_sides - math.pi / 2  # Start at top
        x = int(center_x + radius * math.cos(angle))
        y = int(center_y + radius * math.sin(angle))
        points.append((x, y))
    return points

# Draw various polygons
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
shapes = [3, 4, 5, 6, 8, 12]  # Number of sides
names = ['Triangle', 'Square', 'Pentagon', 'Hexagon', 'Octagon', 'Dodecagon']

for idx, (n_sides, name) in enumerate(zip(shapes, names)):
    row = idx // 3
    col = idx % 3
    
    canvas = create_canvas(50, 50)
    points = create_regular_polygon(25, 25, 20, n_sides)
    draw_polygon(canvas, points)
    
    axes[row, col].imshow(canvas, cmap='gray', vmin=0, vmax=1)
    axes[row, col].set_title(f'{name} ({n_sides} sides)', fontsize=11)
    axes[row, col].grid(True, alpha=0.3)
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

print("\nFrom lines to polygons to any shape!")
print("This is the foundation of vector graphics.")

## Section 6: Performance and Historical Context

### Why Does This Algorithm Matter?

When Bresenham developed this algorithm in 1962, computers were radically different from today:
- Memory was measured in kilobytes, not gigabytes
- Floating-point operations were extremely slow
- Integer operations were much faster
- Every instruction counted

Bresenham's algorithm was revolutionary because it replaced slow floating-point calculations with fast integer operations. Let's compare the operations needed:

**Naive approach (per pixel):**
- 1 floating-point multiplication (slow!)
- 1 floating-point addition
- 1 rounding operation

**Bresenham's approach (per pixel):**
- 2 integer additions/subtractions
- 1 integer comparison
- Maybe 1 more subtraction (if decision > 0)

### Modern Relevance

You might wonder: why does this still matter today? Computers are millions of times faster than in 1962!

**It still matters because:**

1. **Graphics hardware**: GPUs draw billions of pixels per second. Even tiny savings matter at that scale.

2. **Embedded systems**: Many devices (smartwatches, IoT sensors, medical devices) have limited computing power.

3. **Power efficiency**: Integer operations use less power than floating-point, important for battery life.

4. **Elegance**: The algorithm is a beautiful example of problem-solving through mathematical insight.

### The Broader Impact

Bresenham's algorithm inspired similar approaches for:
- Circle drawing (midpoint circle algorithm)
- Ellipse drawing
- Curve rendering
- 3D line projection

The core insight—replacing floating-point operations with integer error tracking—became a fundamental technique in computer graphics.

## Section 7: Exercises and Challenges

Now it's your turn to experiment with what we've learned!

### Exercise 1: Create a Grid

Use Bresenham's algorithm to draw a grid of lines (like graph paper). Can you make:
- A grid with 10-pixel spacing?
- A grid where some lines are thicker (to mark every 50 pixels)?
- A radial grid (lines emanating from center)?

In [ ]:
# Your code here
canvas = create_canvas(100, 100)

# Draw a grid...

# show_canvas(canvas, "My Grid")

### Exercise 2: Star Patterns

Create interesting patterns by drawing lines:
- A five-pointed star
- A spiral of lines
- A sunburst pattern
- Concentric polygons

In [ ]:
# Your code here
canvas = create_canvas(100, 100)

# Create a pattern...

# show_canvas(canvas, "My Pattern")

### Exercise 3: Animation

Can you create a simple animation? For example:
- A line rotating around a point
- A line growing longer
- Multiple lines forming and dissolving

Hint: Use a loop that creates frames, showing each frame briefly.

In [ ]:
# Animation example
import time
from IPython.display import clear_output

# Your animation code here...
# for frame in range(num_frames):
#     canvas = create_canvas(50, 50)
#     # Draw something that changes with frame...
#     clear_output(wait=True)
#     show_canvas(canvas, f"Frame {frame}")
#     time.sleep(0.1)

### Challenge: Optimize Further

Can you make Bresenham's algorithm even faster?

Ideas to explore:
- Pre-compute 2×dx and 2×dy to avoid multiplication in the loop
- Use bit-shifting instead of multiplication by 2 (x << 1 instead of 2×x)
- Unroll the loop for common cases

Measure the performance difference!

## Section 8: Reflection and Looking Ahead

### What Have We Learned?

In this notebook, we've discovered Bresenham's elegant solution to line drawing:

**The core algorithm**: By tracking a decision variable that represents accumulated error, we can determine when to increment y (for gentle lines) or x (for steep lines) using only integer arithmetic.

**Generalization**: The same basic approach works for lines in all eight octants by adjusting step directions and which coordinate we primarily increment.

**Extensions**: The algorithm can be modified to create thick lines, patterned lines, and serves as the foundation for drawing polygons and more complex shapes.

**Historical significance**: This 1962 algorithm is still relevant today because of its efficiency, elegance, and the fundamental insights it embodies.

### The Bigger Picture

Bresenham's algorithm exemplifies several important principles in computer science:

**Mathematical insight solves practical problems**: By understanding the geometry of the problem, Bresenham found a way to eliminate expensive operations.

**Constraints drive innovation**: The limitations of 1960s hardware led to an algorithm that's still valuable decades later.

**Simple building blocks enable complexity**: Lines seem basic, but they're the foundation for all vector graphics, from simple shapes to complex illustrations.

**Efficiency matters at scale**: When you're drawing millions of lines per second, these optimizations make a real difference.

### Looking Forward

With lines mastered, we're ready to tackle curves! In the next notebooks, we'll explore:

**Part 2: Circles and Arcs**
- Midpoint circle algorithm (Bresenham's method applied to circles!)
- Drawing arcs and ellipses
- Filled vs. outline shapes

**Part 3: Bézier Curves**
- Parametric curve equations
- Rasterizing smooth curves
- The math behind fonts and vector graphics

**Part 4: Anti-Aliasing**
- Making lines and curves look smooth
- Xiaolin Wu's algorithm
- The perception vs. reality of pixels

### Questions for Reflection

Before moving on, consider:

1. How might you extend Bresenham's approach to 3D lines?
2. What other geometric problems might benefit from integer-only algorithms?
3. How do modern GPUs handle line drawing differently?
4. What trade-offs exist between speed, quality, and simplicity in graphics algorithms?

## Additional Resources

To explore these topics further:

**Primary Sources:**
- [Bresenham's original 1965 IBM paper](https://www.cs.helsinki.fi/group/goa/mallinnus/lines/bresenh.html)
- [Interview with Jack Bresenham](http://www.computerhistory.org/collections/catalog/102738327) - Oral history from Computer History Museum

**Algorithms and Theory:**
- [Line Drawing Algorithms](https://en.wikipedia.org/wiki/Line_drawing_algorithm) - Wikipedia overview
- [Michael Abrash's Graphics Programming Black Book](https://www.jagregory.com/abrash-black-book/) - Chapter 35-38 cover line drawing
- [Computer Graphics: Principles and Practice](https://www.cgpp.net/) - The definitive textbook

**Interactive Demos:**
- [Bresenham's Algorithm Visualization](https://www.cs.helsinki.fi/group/goa/mallinnus/lines/bresenh.html)
- [Drawing Lines is Hard](https://medium.com/@phinjensen/drawing-lines-is-hard-f5b70e9b0016) - Interesting blog post on complexities

**Modern Context:**
- [GPU Rendering Pipeline](https://www.khronos.org/opengl/wiki/Rendering_Pipeline_Overview) - How modern graphics hardware works
- [Fixed-Point Mathematics](https://en.wikipedia.org/wiki/Fixed-point_arithmetic) - Alternative to floating-point

---

**Next:** [Part 2A: Circle Drawing Fundamentals](part2a_circle_fundamentals.ipynb)

In Part 2A, we'll discover how to draw perfect circles using similar integer-only techniques, exploring the beauty of circular symmetry and the challenges of rasterizing curves.